# Exploring PM2.5 and PM10 Air Quality Trends in New York–Newark–Jersey City (NY-NJ Metropolitan Area)

**Author:** Sheila Alonso  
**Last updated:** April 2026  
**Data source:** [EPA Air Quality System (AQS)](https://www.epa.gov/outdoor-air-quality-data)

## Introduction

This project presents a data analysis of air quality in the New York–Newark–Jersey City (NY-NJ) Metropolitan Area, focusing on two atmospheric pollutants:

- **PM2.5** — Fine particulate matter with diameter ≤ 2.5 micrometers. Due to its small size, it can penetrate deep into the lungs and enter the bloodstream.
- **PM10** — Coarser particles with diameter ≤ 10 micrometers. These can irritate the respiratory system.

Both pollutants are regulated by the [U.S. Environmental Protection Agency (EPA)](https://www.epa.gov/outdoor-air-quality-data) and monitored continuously across multiple stations in the metropolitan area.

> **Note on data availability:** PM2.5 data covers 2020–April 2026. PM10 data covers 2020–2025, as 2026 data is not yet available from the EPA source at the time of this analysis.

## Project Overview

### Objectives
- Analyze PM2.5 and PM10 air quality trends in the NY-NJ Metropolitan Area from 2020 to 2026.
- Identify seasonal patterns and year-over-year changes.
- Detect episodes of elevated pollution and examine their timing.
- Compare PM2.5 and PM10 behaviour over the same period.

### Research Questions
1. How have PM2.5 and PM10 levels changed over time?
2. Are there consistent seasonal patterns in pollutant concentrations?
3. Can episodes of high pollution be identified and characterised?
4. What is the relationship between PM2.5 and PM10 concentrations?

### Data Source
All data was downloaded from the [EPA Air Quality System (AQS) Data Mart](https://www.epa.gov/outdoor-air-quality-data), which provides daily measurements from regulatory monitoring stations across the United States.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

# Style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("Setup complete.")

## 1. Data Loading

Data downloaded from the [EPA AQS Data Mart](https://www.epa.gov/outdoor-air-quality-data).  
CSV files are stored in the `data/` folder relative to this notebook.

- **PM2.5:** 2020–2026 (2026 through April)  
- **PM10:** 2020–2025 (2026 data not yet available from EPA)

In [ ]:
DATA_DIR = Path("data")

PM25_YEARS = range(2020, 2027)  # 2020 to 2026
PM10_YEARS = range(2020, 2026)  # 2020 to 2025

def load_datasets(pollutant, years, data_dir=DATA_DIR):
    """Load yearly CSVs for a given pollutant into a dictionary {year: DataFrame}."""
    datasets = {}
    for y in years:
        fpath = data_dir / f"{pollutant}_{y}_NewYork.csv"
        if fpath.exists():
            datasets[y] = pd.read_csv(fpath)
            print(f"  ✔ {fpath.name}  ({len(datasets[y]):,} rows)")
        else:
            print(f"  ✗ Missing: {fpath.name}")
    return datasets

print("Loading PM2.5 datasets:")
PM25_files = load_datasets("pm_2_5", PM25_YEARS)

print("\nLoading PM10 datasets:")
PM10_files = load_datasets("pm_10", PM10_YEARS)

## 2. Data Cleaning

Each yearly file may use slightly different column names. The `normalize_df` function handles this automatically, extracting only the date and concentration value, converting types, and removing invalid records.

In [ ]:
def normalize_df(df, value_candidates, date_candidates=("date_local", "Date")):
    """
    Normalize a raw EPA DataFrame to two columns: date (datetime) and value (float).
    
    Parameters
    ----------
    df : pd.DataFrame
    value_candidates : list[str]  Column names to try for the concentration value.
    date_candidates  : tuple[str] Column names to try for the date.
    """
    date_col = next((c for c in date_candidates if c in df.columns), None)
    val_col  = next((c for c in value_candidates if c in df.columns), None)
    if date_col is None or val_col is None:
        return pd.DataFrame(columns=["date", "value"])
    
    out = df[[date_col, val_col]].copy()
    out.columns = ["date", "value"]
    out["date"]  = pd.to_datetime(out["date"], dayfirst=False)
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out = out.dropna(subset=["value"])
    out = out[out["value"] >= 0]   # remove physically impossible negatives
    return out


# Concatenate all years into single DataFrames
pm25_raw = (
    pd.concat(
        [normalize_df(df, ["Daily Mean PM2.5 Concentration"]) for df in PM25_files.values()],
        ignore_index=True
    ).sort_values("date").reset_index(drop=True)
)

pm10_raw = (
    pd.concat(
        [normalize_df(df, ["Daily Mean PM10 Concentration"]) for df in PM10_files.values()],
        ignore_index=True
    ).sort_values("date").reset_index(drop=True)
)

print(f"PM2.5 — {len(pm25_raw):,} records  |  {pm25_raw['date'].min().date()} → {pm25_raw['date'].max().date()}")
print(f"PM10  — {len(pm10_raw):,} records  |  {pm10_raw['date'].min().date()} → {pm10_raw['date'].max().date()}")
print(f"\nPM2.5 missing values: {pm25_raw['value'].isna().sum()}")
print(f"PM10  missing values: {pm10_raw['value'].isna().sum()}")

## 3. Aggregations

From the raw daily records (multiple monitoring stations per day), we compute:
- **Daily mean** across all stations
- **7-day rolling average** to smooth short-term variability
- **Monthly mean** for trend analysis

In [ ]:
def daily_with_rolling(df, window=7):
    """Compute daily mean and 7-day rolling average."""
    s = (
        df.groupby("date", as_index=False)["value"]
        .mean()
        .rename(columns={"value": "daily"})
    )
    s["roll"] = s["daily"].rolling(window, min_periods=1).mean()
    return s


def monthly_mean(df):
    """Aggregate raw data to monthly means, adding year/month/abbr columns."""
    m = df.copy()
    m["year"]       = m["date"].dt.year
    m["month"]      = m["date"].dt.month
    m["month_abbr"] = m["date"].dt.strftime("%b")
    return (
        m.groupby(["year", "month", "month_abbr"], as_index=False)["value"]
        .mean()
        .rename(columns={"value": "mean"})
    )


pm25_daily   = daily_with_rolling(pm25_raw)
pm10_daily   = daily_with_rolling(pm10_raw)
pm25_monthly = monthly_mean(pm25_raw)
pm10_monthly = monthly_mean(pm10_raw)

print("Aggregations ready.")
print(f"  PM2.5 daily:   {len(pm25_daily)} days")
print(f"  PM10  daily:   {len(pm10_daily)} days")

## 4. Descriptive Statistics

In [ ]:
stats = pd.DataFrame({
    "PM2.5": pm25_raw["value"].describe(),
    "PM10":  pm10_raw["value"].describe()
}).round(2)

print(stats)

## 5. Time Series Analysis

### 5.1 Daily trends with 7-day rolling average

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, daily, label, color in zip(
    axes,
    [pm25_daily, pm10_daily],
    ["PM2.5 (µg/m³)", "PM10 (µg/m³)"],
    ["steelblue", "coral"]
):
    ax.fill_between(daily["date"], daily["daily"], alpha=0.25, color=color)
    ax.plot(daily["date"], daily["roll"], color=color, linewidth=1.8, label="7-day rolling avg")
    ax.set_ylabel(label, fontsize=11)
    ax.legend(fontsize=10)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.suptitle("PM2.5 and PM10 Daily Concentrations — NY-NJ Metro Area (2020–2026)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 5.2 Monthly means by year

In [ ]:
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, monthly, title in zip(
    axes,
    [pm25_monthly, pm10_monthly],
    ["PM2.5 Monthly Mean by Year", "PM10 Monthly Mean by Year"]
):
    for year, grp in monthly.groupby("year"):
        grp = grp.sort_values("month")
        ax.plot(grp["month"], grp["mean"], marker="o", label=str(year))
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_LABELS, rotation=45)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Month")
    ax.set_ylabel("Mean Concentration (µg/m³)")
    ax.legend(title="Year", fontsize=9)

plt.tight_layout()
plt.show()

### 5.3 Average seasonal profile (all years)

In [ ]:
def seasonal_profile(monthly):
    return monthly.groupby("month", as_index=False)["mean"].mean().sort_values("month")

p25_season = seasonal_profile(pm25_monthly)
p10_season = seasonal_profile(pm10_monthly)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(p25_season["month"], p25_season["mean"], marker="o", label="PM2.5", color="steelblue")
ax.plot(p10_season["month"], p10_season["mean"], marker="s", label="PM10",  color="coral")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MONTH_LABELS, rotation=45)
ax.set_title("Average Seasonal Profile — PM2.5 vs PM10 (all years combined)", fontsize=12)
ax.set_xlabel("Month")
ax.set_ylabel("Mean Concentration (µg/m³)")
ax.legend()

plt.tight_layout()
plt.show()

### 5.4 Relationship between PM2.5 and PM10

In [ ]:
cmp = pd.merge(
    pm25_daily[["date", "daily"]].rename(columns={"daily": "pm25"}),
    pm10_daily[["date", "daily"]].rename(columns={"daily": "pm10"}),
    on="date", how="inner"
).dropna()

cmp["year"] = pd.to_datetime(cmp["date"]).dt.year

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(cmp["pm25"], cmp["pm10"], c=cmp["year"], cmap="viridis", alpha=0.5, s=20)
plt.colorbar(scatter, ax=ax, label="Year")

# Regression line
m, b = np.polyfit(cmp["pm25"], cmp["pm10"], 1)
x_line = np.linspace(cmp["pm25"].min(), cmp["pm25"].max(), 100)
ax.plot(x_line, m * x_line + b, color="red", linewidth=2, label=f"y = {m:.2f}x + {b:.2f}")

corr = cmp[["pm25", "pm10"]].corr().iloc[0, 1]
ax.set_title(f"PM2.5 vs PM10 Daily Means  (r = {corr:.2f})", fontsize=12)
ax.set_xlabel("PM2.5 (µg/m³)")
ax.set_ylabel("PM10 (µg/m³)")
ax.legend()

plt.tight_layout()
plt.show()

print(f"Pearson correlation: {corr:.3f}")

## 6. Interactive Visualisations (Bokeh)

The following charts are interactive — hover for values, scroll to zoom, drag to pan.

In [ ]:
# --- 6.1 Daily time series ---
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, daily, label, color in zip(
    axes,
    [pm25_daily, pm10_daily],
    ["PM2.5 (µg/m³)", "PM10 (µg/m³)"],
    ["steelblue", "coral"]
):
    ax.fill_between(daily["date"], daily["daily"], alpha=0.2, color=color)
    ax.plot(daily["date"], daily["roll"], color=color, linewidth=2,
            label="7-day rolling avg")
    ax.set_ylabel(label, fontsize=11)
    ax.legend(fontsize=10)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.suptitle("PM2.5 and PM10 — Daily Values and 7-day Rolling Average", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6.2 Monthly comparison by year ---
years = sorted(set(pm25_monthly["year"]).intersection(pm10_monthly["year"]))
n_years = len(years)
n_cols = 4
n_rows = (n_years + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4), sharey=False)
axes = axes.flatten()

for idx, y in enumerate(years):
    ax = axes[idx]
    a = pm25_monthly[pm25_monthly["year"] == y].sort_values("month")
    b = pm10_monthly[pm10_monthly["year"] == y].sort_values("month")
    ax.plot(a["month"], a["mean"], marker="o", color="steelblue",
            linewidth=2, label="PM2.5")
    ax.plot(b["month"], b["mean"], marker="s", color="coral",
            linewidth=2, label="PM10")
    ax.set_title(str(y), fontsize=11)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_LABELS, rotation=45, fontsize=8)
    ax.set_ylabel("µg/m³", fontsize=9)
    ax.legend(fontsize=8)

# Hide unused subplots
for idx in range(len(years), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle("Monthly Means by Year — PM2.5 and PM10", fontsize=13)
plt.tight_layout()
plt.show()

## 7. High Pollution Episodes

The EPA 24-hour standard for PM2.5 is **35 µg/m³**. Days exceeding this threshold are considered unhealthy for sensitive groups or worse.  
For PM10 the standard is **150 µg/m³**.

In [ ]:
EPA_PM25_THRESHOLD = 35
EPA_PM10_THRESHOLD = 150

pm25_exceed = pm25_daily[pm25_daily["daily"] > EPA_PM25_THRESHOLD].copy()
pm10_exceed = pm10_daily[pm10_daily["daily"] > EPA_PM10_THRESHOLD].copy()
pm25_exceed["year"] = pd.to_datetime(pm25_exceed["date"]).dt.year

print(f"PM2.5 days exceeding {EPA_PM25_THRESHOLD} µg/m³: {len(pm25_exceed)}")
print(f"PM10  days exceeding {EPA_PM10_THRESHOLD} µg/m³: {len(pm10_exceed)}")
exceed_by_year = pm25_exceed.groupby("year").size().reset_index(name="days_above_threshold")
print("PM2.5 exceedance days per year:")
print(exceed_by_year.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(pm25_daily["date"], pm25_daily["daily"], alpha=0.2, color="steelblue")
ax.plot(pm25_daily["date"], pm25_daily["roll"], color="steelblue", linewidth=1.5, label="7-day rolling avg")
ax.scatter(pm25_exceed["date"], pm25_exceed["daily"], color="red", s=15, zorder=5, label=f"Exceeds EPA threshold ({EPA_PM25_THRESHOLD} µg/m³)")
ax.axhline(EPA_PM25_THRESHOLD, color="red", linestyle="--", linewidth=1, alpha=0.7)
ax.set_title("PM2.5 Daily Concentrations — EPA Threshold Exceedances Highlighted", fontsize=12)
ax.set_ylabel("PM2.5 (µg/m³)")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend()

plt.tight_layout()
plt.show()

print("Top 10 worst PM2.5 days:")
print(pm25_daily.nlargest(10, "daily")[["date", "daily", "roll"]].to_string(index=False))

## 8. Monthly Heatmap

A year x month heatmap provides an at-a-glance view of seasonal patterns and anomalous years.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, monthly, title in zip(axes, [pm25_monthly, pm10_monthly],
    ["PM2.5 Monthly Mean Heatmap (µg/m³)", "PM10 Monthly Mean Heatmap (µg/m³)"]):
    pivot = monthly.pivot(index="year", columns="month", values="mean")
    pivot.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][:len(pivot.columns)]
    sns.heatmap(pivot, ax=ax, cmap="YlOrRd", annot=True, fmt=".1f", linewidths=0.5, cbar_kws={"label": "µg/m³"})
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Month")
    ax.set_ylabel("Year")
    
plt.tight_layout()
plt.show()

## 9. Annual Summary Table

In [ ]:
def annual_summary(daily_df, threshold, label):
    df = daily_df.copy()
    df["year"] = pd.to_datetime(df["date"]).dt.year
    summary = df.groupby("year").agg(
        mean=("daily", "mean"),
        median=("daily", "median"),
        max=("daily", "max"),
        days_above=("daily", lambda x: (x > threshold).sum())
    ).round(2)
    summary.columns = ["Mean", "Median", "Max", f"Days > {threshold} µg/m³"]
    return summary

print(f"PM2.5 Annual Summary (threshold: {EPA_PM25_THRESHOLD} µg/m³):")
print(annual_summary(pm25_daily, EPA_PM25_THRESHOLD, "PM2.5").to_string())
print()
print(f"PM10 Annual Summary (threshold: {EPA_PM10_THRESHOLD} µg/m³):")
print(annual_summary(pm10_daily, EPA_PM10_THRESHOLD, "PM10").to_string())

## 10. Conclusions

Based on the analysis of PM2.5 and PM10 data from 2020 to 2026 in the NY-NJ Metropolitan Area:

---

**1. PM2.5 shows a consistent seasonal pattern.**

Higher concentrations occur in winter months (December–January) and lower values in summer, likely driven by heating emissions and atmospheric conditions that trap pollutants near ground level.

---

**2. PM10 data is sparser.**

Fewer monitoring stations measure PM10 in this area, but the available data follows a similar seasonal trend, with elevated values in colder months.

---

**3. Year-over-year variability exists.**

Some years show higher baseline peaks — notably 2020 early months, a period when lockdowns significantly altered traffic and industrial activity patterns.

---

**4. PM2.5 and PM10 are moderately correlated.**

They share common sources (vehicle traffic, industrial emissions) but behave independently during events like dust storms or wildfires, which tend to raise PM10 disproportionately relative to PM2.5.

---

**5. 2026 data (PM2.5 only, through April)** continues the established seasonal pattern, with no anomalous spikes observed so far.

---

**6. The anomalous 2023 spike was caused by the Canadian wildfires.**

Between June 6 and 8, 2023, wildfires in Quebec, Canada generated massive smoke plumes transported to the NY-NJ area by a low-pressure system stalled over the Gulf of Maine. Its counterclockwise rotation directed winds and smoke directly toward New York. On June 7, the EPA recorded a 24-hour average PM2.5 level of approximately 100 µg/m³ in New York City — about 10 times the annual average — while emergency department visits for asthma-related symptoms jumped to 261 per day compared to a baseline of around 182. This event illustrates how meteorological factors (wind direction, pressure systems) can transport pollutants hundreds of kilometres from their source, causing spikes that break the usual seasonal patterns.

---

### Limitations
- PM10 monitoring stations in the NY-NJ metro area are fewer than PM2.5 stations, which may introduce sampling bias.
- 2026 PM10 data is not yet available from the EPA source.
- This analysis focuses on daily means across all stations; station-level spatial analysis could reveal localised pollution hotspots.

### References
- U.S. Environmental Protection Agency — Air Quality System (AQS) Data Mart: https://www.epa.gov/outdoor-air-quality-data
- Chen et al. (2023). Canadian Wildfire Smoke and Asthma Syndrome Emergency Department Visits in New York City. *JAMA*, 330(14), 1385–1387. https://pmc.ncbi.nlm.nih.gov/articles/PMC10514869/
- Communications Earth & Environment (2025). Radiative cooling in New York/New Jersey metropolitan areas by wildfire particulate matter from the 2023 Canadian wildfires. https://www.nature.com/articles/s43247-025-02214-3
- NYC Community Air Survey (2023). https://a816-dohbesp.nyc.gov/IndicatorPublic/data-features/nyccas/